# MPPP — 03 COLMAP alignment (Navcam, optionally + Mastcam-Z 34 mm)

This notebook takes PDS products to a COLMAP database and a refined, prior-aligned reconstruction. All settings are in section 0, which is tagged `parameters`; `scripts/run_scapes.py` runs this notebook for several sites by overriding them.

**Sites.** Pick a named site from `SITES` or add your own sol range:

| site | sols |
|---|---|
| Three Forks (default test case) | 684–693 |
| Three Forks, larger | 670–694 |
| Belva | 748–815 |
| Rockytop | 461–530 |
| Bell Island | 1451–1467 |
| Taylor Fjellet | 1601–1646 |
| landing site | 9–48 |

**Mastcam-Z.** With `INCLUDE_ZCAM34 = True`, the Mastcam-Z 34 mm frames (`ZL0`/`ZR0`, `_034`) of the same sols join the block. Each eye is split into cameras by focus motor count (`ZCAM_FOCUS_BIN`), and each bin refines its own focal length. Since v0p22 the bins start from the focal-length-against-focus model of earlier solutions (`ZCAM_INTRINSICS = "focus_model"`); bins with at most `ZCAM_HOLD_F_IMAGES` images hold their focal length there.

**Steps:**

1. **Select**: one product per exposure (the largest, then the highest version).
2. **Process** with MPPP: 8-bit PNG, masks, intrinsics, and CAHV + waypoint poses, with frames padded to the detector frame. `REPROCESS_ALL = True` processes every image again; otherwise images processed with the same configuration are reused.
3. **Project**: one COLMAP camera per eye at full resolution.
   - The rig rotation starts from the refined consensus (`NAVCAM_RIG = "consensus"`). The rig translation, the stereo baseline, is the CAHV value and stays fixed, because it sets the scale.
   - Navcam uses the rational lens model (`M2020_N{L,R}_rational.json`, a consensus of earlier solutions), valid to the frame corners.
4. **Features** at native resolution with masks, then the **database**.
5. **Matching**: exhaustive, with the SIFT and verification settings in `MATCH` (since v0p22 the SIFT ratio test is 0.9; COLMAP's default is 0.8). DSP-SIFT and affine-adapted SIFT are available in `SIFT`.
6. **Reconstruct**: triangulation from the CAHV poses (settings in `TRIANGULATION`), then a weighted bundle adjustment with position and attitude priors.
   - `EXCLUDE_OUTLIERS` (v0p22) drops frames that do not align with the majority, then re-adjusts.
   - `NAVCAM_INTRINSICS`: `"refine"` (default) refines the Navcam intrinsics in every block; `"hold"` keeps the start cameras; `"auto"` holds them where the network is weak (fewer than `NETWORK_MIN_STATIONS` stations or less than `NETWORK_MIN_SPAN_M` of span), since a weak network cannot observe them. The network verdict is printed either way. `NAVCAM_RIG_REFINE` does the same for the rig rotation.
   - `STAGED` (v0p22.2): with Mastcam-Z in the project, the Navcam block is solved first on its own and exported (`error_input_navcam/`); the Mastcam-Z frames are then added with the Navcam cameras and rig held. The Navcam-only and the combined solutions then come from the same images, features and database and are directly comparable.
   - `NAVCAM_CAMERAS` (v0p22.2): a folder of verified consensus Navcam cameras written by notebook 05 to start from, instead of the shipped ones.
   - `ADD_NEARBY_WAYPOINTS` (v0p22.4): every other visit within that many metres of a station of the sol range joins the block. `SKY_ELEVATION_DEG`: Navcam frames pointing higher than this (sky surveys) are left out before processing.
   - The tie points are also counted by convergence angle.
7. **Alignment health**: check this before using the result.
8. **Export** `error_input/` for notebook 04 (error analysis) and notebook 05 (camera models).

This kernel needs pip `pycolmap` 4.2 and `pyceres` 2.6 (`pip install -e .[sfm]`).

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import sys, json
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import mppp
import pycolmap, pyceres
from mppp.select import find_imgs
from mppp.sfm import (SfmProject, select_best_products, extract_features, build_database, prior_overlap_pairs,
                      match, reconstruct, export_for_error, check_gpu_python, check_ba_environment)
from mppp.sfm.health import assess_alignment, health_table, write_health
from mppp.sfm.reconstruction import convergence_statistics
print("MPPP", mppp.__version__, "from", Path(mppp.__file__).parent, "| pycolmap", pycolmap.__version__,
      "(CUDA:", pycolmap.has_cuda, ") | pyceres", pyceres.__version__)
check_ba_environment()        # stops here if this kernel cannot run the weighted BA (e.g. it is the conda CUDA env)

## 0. Settings
Every setting of the run is in the cell below. `scripts/run_scapes.py` overrides them, one site at a time.

In [ ]:
# --- data and output ------------------------------------------------------------------------------------
PDS_DIR     = Path("D:/data/m2020")          # the local PDS archive (searched recursively)
SCAPES_ROOT = Path("D:/scapes/colmap")       # one WORK folder per site below this
# Optional GPU for feature extraction and matching (Windows): the python.exe of a second environment with the
# conda-forge CUDA pycolmap of the same COLMAP version (README, "GPU on Windows").  None = this kernel, CPU.
GPU_PY = None                                # e.g. r"C:\Users\<you>\AppData\Local\miniconda3\envs\mppp_gpu\python.exe"

# --- site ---------------------------------------------------------------------------------------------------
SITES = {                                    # sol range of the Navcam (+ Mastcam-Z) block; add your own
    "butler_landing":       (2, 14),
    # "sid":                (360, 380),
    "rockytop":             (461, 530),
    "threeforks":           (685, 692),
    "threeforks_large":     (652, 693),
    "belva_crater":         (784, 815),
    "tuxedo_park":          (897, 908),
    "airey_hill":           (961, 992),
    "bunsen_peak":          (1066, 1095),
    "pearce_canyon":        (1183, 1218),
    "rio_chiquito":         (1334, 1340),
    "south_arm":            (1410, 1414),
    "bell_island":          (1451, 1467),
    "taylorfjellet":        (1601, 1645),
    "olifants":             (1775, 1813),
    "groloy":               (1922, 1934),
    "marble_mountain":      (1965, 1979),
}
SITE = "rockytop"           # difficult case (winter sun, low cross-station match rate)
# SITE = "threeforks"       # easy case, flat ground
# SITE = "airey_hill"       # three stations
# SITE = "belva_crater"     #
# SITE = "pearce_canyon"    # scientifically interesting
# SITE = "bell_island"      # cross-station packing
# SITE = "taylorfjellet"    # LMST time range
# SITE = "olifants"         #
INCLUDE_ZCAM34 = False                       # add the Mastcam-Z 34 mm frames (ZL0/ZR0 "_034") of the same sols
ADD_NEARBY_WAYPOINTS = 5                     # v0p22.4: also take every waypoint station within this many metres
                                             # (inclusive) of a station imaged in the sol range, any sol; 0 = off

# --- processing ---------------------------------------------------------------------------------------------
REPROCESS_ALL        = False   # True: process every image again (new masks, radiometry, poses); False: reuse
KEEP_ONLY_REMAINING  = True    # True: only the images still in processed/images_png8 (delete unsuitable ones
                               # there); on a first run, with no folder yet, everything is processed
SKY_ELEVATION_DEG    = 45.0    # v0p22.4: Navcam frames pointing higher than this (sky surveys) are not processed
NO_MASK_INFERENCE_AT = [       # stations where the mask model is off (the rover stays in the images)
                            # "S003D0000",    # OEB landing site
                            # "S026D1004",    # Rockytop
                            "S032D1184",      # Three Forks
                            # "S047D0000",    # Airey Hill
                        ]
STORE_MASK_IN_ALPHA  = True    # also put the mask in the PNG alpha channel (masks/ is written either way)

# --- cameras ------------------------------------------------------------------------------------------------
NAVCAM_DISTORTION = "rational"     # "rational" (valid to the frame corners) or "polynomial" (Metashape K1-K3)
NAVCAM_RIG        = "consensus"    # rig rotation start: "consensus" (refined, v0p22) or "cahv"; baseline = CAHV
NAVCAM_CAMERAS    = None           # v0p22.2: folder of verified consensus Navcam cameras from notebook 05 (None: shipped)
TANGENTIAL        = "refine"       # p1, p2: "refine", "xml" (hold at the calibration) or "zero"
ZCAM_INTRINSICS   = "focus_model"  # Mastcam-Z start: "focus_model" (v0p22), "label" or "xml"
ZCAM_FOCUS_BIN    = 30             # focus motor counts per Mastcam-Z camera (None: one camera per eye)
ZCAM_BIN_REFINE   = "focal"        # a focus-bin camera refines fx, fy only ("all": also principal point, distortion)
ZCAM_HOLD_F_IMAGES = 2             # focus bins with <= this many images hold f at the focus model
ZCAM_RIG          = True           # True: the two Mastcam-Z eyes of an exposure form a stereo rig; False: each
                                   # image is its own frame

# --- features and matching ----------------------------------------------------------------------------------
MAX_NUM_FEATURES = 12000
SIFT = dict(domain_size_pooling=False, estimate_affine_shape=False)   # DSP-SIFT / affine shape (CPU only, slower)
MATCH_MODE = "exhaustive"          # "exhaustive" (baseline) or "prior_pairs" (quick / CPU)
MATCH = dict(max_ratio=0.9, max_distance=0.7, cross_check=True, guided_matching=False, max_error_px=6.0)

# --- reconstruction -----------------------------------------------------------------------------------------
# rounds of (triangulation threshold [full-res px], Cauchy scale [sigma], max residual kept [native px])
SCHEDULE = ((24.0, 10.0, 8.0), (12.0, 4.0, 4.0), (8.0, 2.0, 2.0), (8.0, 2.0, 2.0))
MIN_TRI_ANGLE_DEG = 0.25           # keeps stereo-only points out to ~97 m at the 0.42 m baseline
TRIANGULATION = dict(max_transitivity=1)     # COLMAP track building (see mppp.sfm.reconstruction.triangulate)
ATTITUDE_PRIOR_DEG = 1.0           # weak CAHV attitude prior per frame
EXCLUDE_OUTLIERS = True            # v0p22: drop frames that do not align with the majority, then re-adjust
OUTLIER_THRESHOLDS = {}            # e.g. {"residual_factor": 3.0, "min_shift_m": 0.25} (reconstruction.OUTLIER_DEFAULTS)
NAVCAM_INTRINSICS = "refine"       # "refine" (default: always refine the Navcam intrinsics), "hold" (keep the start
                                   # cameras), or "auto" (hold where the network is weak: < 4 stations or < 5 m span)
NAVCAM_RIG_REFINE = "rotation"     # v0p22.2: "rotation" (refine the rig rotation; the baseline is always held) or "hold"
STAGED            = True           # v0p22.2: Navcam block first, then Mastcam-Z with the Navcam cameras and rig held
HOLD_CAMERAS      = ()             # further camera keys to hold at their start values, e.g. ("ZL034_f0",)

In [ ]:
# derived settings
SOLS = SITES[SITE]
WORK = SCAPES_ROOT / ((f"{SITE}_colmap_nav_zcam34" if INCLUDE_ZCAM34 else f"{SITE}_colmap")
                      + (f"_near{ADD_NEARBY_WAYPOINTS:g}m" if ADD_NEARBY_WAYPOINTS else ""))
ZERO_TERMS = ("p1", "p2", "b1", "b2") if TANGENTIAL == "zero" else ("b1", "b2")
if GPU_PY:
    print("GPU env:", check_gpu_python(GPU_PY))
print(SITE, SOLS, "->", WORK, "| Mastcam-Z 34 mm:", bool(INCLUDE_ZCAM34), "| reprocess all:", bool(REPROCESS_ALL))

## 1–2. Select and process

In [ ]:
if ADD_NEARBY_WAYPOINTS:              # the sol range plus other visits within ADD_NEARBY_WAYPOINTS metres of its stations
    from mppp.select import find_imgs_near
    WAYPOINTS = mppp.load_waypoints()
    paths, near = find_imgs_near(PDS_DIR, ["NLF", "NRF"], SOLS, WAYPOINTS, radius_m=ADD_NEARBY_WAYPOINTS)
    print(f"Navcam: {near['n_in_range']} in sols {SOLS[0]}-{SOLS[1]} + {near['n_added']} from other visits within "
          f"{ADD_NEARBY_WAYPOINTS} m")
    for a in near["stations_added"]:
        print(f"   {a['station']} ({a['distance_m']} m from {a['nearest']}): {a['images']} images, sols {a['sols'][0]}-{a['sols'][1]}")
else:
    paths = find_imgs(PDS_DIR, ["NLF", "NRF"], sol_range=SOLS)
prefixes = ("NCAM",)
if INCLUDE_ZCAM34:
    if ADD_NEARBY_WAYPOINTS:
        zp, znear = find_imgs_near(PDS_DIR, ["ZL0", "ZR0"], SOLS, WAYPOINTS, radius_m=ADD_NEARBY_WAYPOINTS, sequ_id="_034")
        print(f"Mastcam-Z 34: {znear['n_in_range']} in range + {znear['n_added']} from other visits")
        paths += zp
    else:
        paths += find_imgs(PDS_DIR, ["ZL0", "ZR0"], sol_range=SOLS, sequ_id="_034")
    prefixes += ("ZCAM",)
paths, sel = select_best_products(paths, sequence_prefix=prefixes)      # one product per exposure
print(json.dumps(sel, indent=1)); print(len(paths), "products")

In [ ]:
cfg = mppp.default_config()
cfg["export"].update(formats=["PNG8"], store_mask_in_alpha=bool(STORE_MASK_IN_ALPHA), write_mask_files=True,
                     write_colmap=False)
# mask model: the released default ("mppp_mask_v2"); or cfg["masking"]["checkpoint"] = "<path to your checkpoint>"
cfg["masking"]["skip_inference_at"] = NO_MASK_INFERENCE_AT
cfg["selection"]["max_boresight_elevation_deg"] = SKY_ELEVATION_DEG      # v0p22.4: sky-pointing Navcam frames are skipped
PROCESSED = WORK / "processed"
# KEEP_ONLY_REMAINING = False: every selected product (images deleted from processed/images_png8 come back).
# REPROCESS_ALL = False: images processed with the same configuration are reused from the manifest.
manifest = mppp.process_images(paths, PROCESSED, cfg, mppp.load_waypoints(), provenance=sel,
                               only_existing="PNG8" if KEEP_ONLY_REMAINING else None,
                               reuse_existing=not REPROCESS_ALL)
print(manifest["n_processed"], "images in the manifest,", len(manifest["failed"]), "failed,",
      len(manifest.get("skipped", [])), "skipped (sky-pointing)")
skipped = [m["source_product"] for m in manifest["images"] if m["mask"].get("inference_skipped")]
if skipped:
    print(len(skipped), "images without mask inference (rover visible):",
          sorted({f"S{m['site']:03d}D{m['drive']:04d}" for m in manifest["images"] if m["mask"].get("inference_skipped")}))
if manifest.get("only_existing"):
    print("kept", manifest["only_existing"]["n_kept"], "of", manifest["only_existing"]["n_selected"], "selected;",
          manifest["only_existing"]["n_removed"], "removed")

## 3–4. Project, features, database

In [ ]:
from collections import Counter
import numpy as np
PROJECT = WORK / "colmap"
proj = SfmProject.load(PROJECT) if (PROJECT / "project.json").is_file() else None
wanted = {"zero_terms": list(ZERO_TERMS), "zcam_intrinsics": ZCAM_INTRINSICS,
          "zcam_focus_bin": float(ZCAM_FOCUS_BIN) if ZCAM_FOCUS_BIN else None, "zcam_bin_refine": ZCAM_BIN_REFINE,
          "zcam_rig": ZCAM_RIG, "navcam_distortion": NAVCAM_DISTORTION, "navcam_rig": NAVCAM_RIG,
          "zcam_hold_f_images": int(ZCAM_HOLD_F_IMAGES) if ZCAM_INTRINSICS == "focus_model" else None,
          "prior_R_corrected": True}
names = {Path(m["source_product"]).stem for m in manifest["images"]}
if (REPROCESS_ALL or proj is None or names != {r["stem"] for r in proj.images}    # other images than the manifest
        or any(proj.settings.get(k) != v for k, v in wanted.items())            # or different camera settings
        or (proj.settings.get("navcam_cameras") or {}).get("dir") != (str(NAVCAM_CAMERAS) if NAVCAM_CAMERAS else None)):
    proj = SfmProject.create(manifest["images"], PROCESSED, PROJECT, prior_sigma_m=(1.0, 1.0, 1.0),
                             zcam_intrinsics=ZCAM_INTRINSICS, zero_terms=ZERO_TERMS,
                             zcam_focus_bin=ZCAM_FOCUS_BIN, zcam_bin_refine=ZCAM_BIN_REFINE, zcam_rig=ZCAM_RIG,
                             navcam_distortion=NAVCAM_DISTORTION, navcam_rig=NAVCAM_RIG,
                             zcam_hold_f_images=ZCAM_HOLD_F_IMAGES,
                             navcam_cameras=NAVCAM_CAMERAS)
else:
    proj.refresh_images(manifest["images"])      # copies (no hard links) follow images processed again
print(len(proj.images), "images;", {k: round(v["baseline_m"], 4) for k, v in proj.rig.items()}, "m baseline;",
      {k: v["n_pairs"] for k, v in proj.rig.items()}, "CAHV pairs;",
      {k: v.get("rotation_source", "CAHV") for k, v in proj.rig.items()}, "rig rotation")
for k, c in proj.cameras.items():
    if not c.get("group"):
        print(k, [round(x, 3) for x in c["params"][:5]], f"p1 {c['params'][6]:.2e} p2 {c['params'][7]:.2e}",
              f"k3 {c['params'][8]:.3g}", f"k4 {c['params'][9]:.3g} (rational)" if c.get("free_params") else "")
bins = Counter(c["group"] for c in proj.cameras.values() if c.get("group"))
for g, n in sorted(bins.items()):
    cs = [c for c in proj.cameras.values() if c.get("group") == g]
    held = sum("fx" in (c.get("fixed_params") or []) for c in cs)
    print(f"{g}: {n} focus bins ({held} with f held), counts {min(c['focus_count_range'][0] for c in cs):.0f}-"
          f"{max(c['focus_count_range'][1] for c in cs):.0f}, images per bin {sorted(c['n_images'] for c in cs)}")
corr = [r["prior_R_correction_deg"] for r in proj.images if "prior_R_correction_deg" in r]
if corr:   # Mastcam-Z label attitudes rotated to fit the camera's principal point (v0p14.5)
    print(f"Mastcam-Z prior attitude correction: median {np.median(corr):.3f} deg, max {max(corr):.3f} deg ({len(corr)} images)")
print("images per camera:", dict(Counter(r["instrument"] for r in proj.images)))

In [ ]:
# SIFT at native resolution with masks -> features.db.  Reused on reruns only if features.json beside it records
# the same settings and image/mask files; overwrite=True always redoes it.
extract_features(proj, max_num_features=MAX_NUM_FEATURES, use_gpu=True if GPU_PY else None, python=GPU_PY,
                 overwrite=bool(REPROCESS_ALL), **SIFT)
print(build_database(proj))          # replaces database.db (and any matches in it) on every run

## 5. Matching

In [ ]:
if MATCH_MODE == "prior_pairs":
    pairs = prior_overlap_pairs(proj, min_overlap=0.05)
    print(len(pairs), "prior pairs")
print(match(proj, mode=MATCH_MODE, use_gpu=True if GPU_PY else None, python=GPU_PY, **MATCH))

## 6. Reconstruct and refine

In [ ]:
from mppp.sfm.reconstruction import navcam_network
net = navcam_network(proj)
print(f"Navcam network: {net['stations']} stations over {net['span_m']:.1f} m, {net['navcam_images']} images -> {net['verdict']}"
      f" (intrinsics setting {NAVCAM_INTRINSICS!r})")
rec = reconstruct(proj, sigma_px=0.5,           # keypoint sigma in native pixels
                  schedule=SCHEDULE, refine_rig=("rotation" if NAVCAM_RIG_REFINE == "rotation" else False),
                  refine_tangential=(TANGENTIAL == "refine"),
                  min_track_length=2,           # two-view tie points (e.g. seen only by one stereo pair) are kept
                  min_tri_angle_deg=MIN_TRI_ANGLE_DEG, attitude_prior_deg=ATTITUDE_PRIOR_DEG,
                  triangulation_options=TRIANGULATION,
                  exclude_outliers=EXCLUDE_OUTLIERS, outlier_thresholds=OUTLIER_THRESHOLDS,
                  navcam_intrinsics=NAVCAM_INTRINSICS, staged=STAGED, hold_cameras=HOLD_CAMERAS)
rs = proj.settings["reconstruction"]
print("Navcam intrinsics:", rs.get("navcam_intrinsics"), "| held cameras:", rs.get("hold_cameras") or "none",
      "| staged:", rs.get("staged"))
excluded = rs.get("excluded", [])
print(len(excluded), "frames excluded as outliers")
for e in excluded:
    print("  ", ", ".join(e["images"]), "|", e["station"], "|", "; ".join(e["reasons"]))
summary = export_for_error(proj, rec)
print(json.dumps({k: summary[k] for k in ("tracks", "residual_rms_native_px", "registered_images", "project_images",
                                           "prior_offset_median_m", "station_components")}, indent=1))
summary_nav = None
if rs.get("navcam_stage"):
    # the stage-1 Navcam-only solution: same images, features and database as the combined one
    import pycolmap
    rec_nav = pycolmap.Reconstruction(str(proj.root / rs["navcam_stage"]["path"]))
    summary_nav = export_for_error(proj, rec_nav, out_dir=proj.root / "error_input_navcam")
    print("Navcam-only stage exported to error_input_navcam/:",
          {k: summary_nav[k] for k in ("tracks", "residual_rms_native_px", "registered_images")})
conv = convergence_statistics(rec, proj)
proj.settings["reconstruction"]["convergence"] = conv
proj.save()
print("tie points by convergence angle", conv["bins_deg"], "deg:", conv["points"],
      "| cross-station:", conv["cross_station_points"], f"| > 10 deg: {conv['points_over_10deg']}, "
      f"> 20 deg: {conv['points_over_20deg']}")
for k, c in proj.cameras.items():
    if c.get("group"):
        continue                                  # Mastcam-Z focus bins: see section 7b
    cid = [r["camera_id"] for r in proj.images if r["instrument"] == k and "camera_id" in r][0]
    p0, p1 = c["params"], rec.cameras[cid].params
    print(k, " ".join(f"{n} {a:.4g}->{b:.4g}" for n, a, b in zip(("fx", "fy", "cx", "cy", "k1", "k2", "p1", "p2", "k3", "k4"),
                                                               p0[:10], p1[:10])))
print("COLMAP GUI: double-click", proj.root / "open_in_colmap.bat")

## 7. Alignment health — check before any analysis
Each check has provisional warn and fail thresholds. These were set a priori and are not validated limits; pass `thresholds={...}` to change them. The report is written to `<project>/health/` (`health.json`, `health.md`, `health.png`).

| section | checks |
|---|---|
| registration | images registered; frames held at their prior (< 30 observations) |
| tie points | points; track-length population (two-view share); observations per image; cross-station ties; tied station blocks |
| reprojection | median and 95th-percentile residual (native px); left vs right eye; outlier images; residual growth towards the image edge; tie-point coverage by image radius |
| camera model | change of f, principal point and k1–k4; image displacement of the same ray |
| stereo rig | right-camera rotation vs its start; spread of the CAHV pairs |
| poses | refined − prior centre per station; spread within a station; attitude change; scale vs the priors |

In [ ]:
report = assess_alignment(proj, rec)
print(health_table(report))
files = write_health(report, proj.root / "health", rec, proj)
from IPython.display import Image, display
display(Image(filename=files["png"]))
if report["verdict"] == "fail":
    print("\nFAIL: fix the alignment (matching, priors, masks, thresholds) before any error analysis.")

### 7b. Mastcam-Z focus breathing
Written to `<project>/health/zcam_focus_breathing.{png,csv,json}`. Notebook 05 compares scapes.

In [ ]:
from mppp.sfm.zcam import write_focus_breathing
if any(c.get("group") for c in proj.cameras.values()):
    fb = write_focus_breathing(proj, rec)
    for g, f in fb["fits"].items():
        for kind in ("refined", "label"):
            if f[kind]:
                print(f"{g} {kind:7s}: f = {f[kind]['f0_px']:.1f} px at count {f['reference_count']:.0f}, "
                      f"slope {f[kind]['slope_px_per_count']:+.4f} px/count, rms {f[kind]['rms_px']:.2f} px, n = {f[kind]['n']}")
    display(Image(filename=fb["png"]))
else:
    print("no Mastcam-Z focus-bin cameras")

## 8. Export for the error analysis
Notebook 04 (`04_error_analysis.ipynb`) reads the `error_input/` folders; notebook 05 compares the camera models.

In [ ]:
import csv
import matplotlib.pyplot as plt
from mppp.sfm.export import plot_camera_shifts
pose_rows = list(csv.DictReader(open(proj.root / "error_input" / "poses.csv")))
fig = plot_camera_shifts(proj, rows=pose_rows, out_png=proj.root / "error_input" / "camera_shifts.png")
plt.show()
if report["verdict"] == "fail":
    print("alignment health is FAIL (section 7): fix the alignment before using error_input/ in notebook 04")
eps = summary.get("eps", {})
print("error_input:", proj.root / "error_input")
print(f"eps (rms residual / sqrt 2): same-station tracks {eps.get('eps_intra_px', float('nan')):.3f} px, "
      f"cross-station tracks {eps.get('eps_cross_px', float('nan')):.3f} px")